# Machine Learning Zoomcamp 2026 - Homework 3: Classification & Logistic Regression

Solución detallada y explicada de la Tarea 3 (Clasificación binaria para Lead Scoring).

### Objetivos:
- Imputación de variables según tipo (categórico -> 'NA', numérico -> 0.0).
- Análisis de variables categóricas (moda) y numéricas (matriz de correlación).
- Análisis de importancia de variables con Mutual Information (`mutual_info_score`).
- Entrenamiento de Regresión Logística con One-Hot Encoding (`DictVectorizer`).
- Selección de características por eliminación iterativa (Feature Elimination).
- Optimización del parámetro de regularización $C$.


## 1. Carga y Preparación de Datos

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, mutual_info_score

data_path = Path("../../data/course_lead_scoring_2026.csv")
df = pd.read_csv(data_path)
TARGET = "converted"

# Preparación de variables
features = df.drop(columns=[TARGET]).copy()
numerical = [c for c in features.columns if pd.api.types.is_numeric_dtype(features[c])]
categorical = [c for c in features.columns if c not in numerical]

features[categorical] = features[categorical].fillna("NA")
features[numerical] = features[numerical].fillna(0.0)

print(f"Dataset shape: {df.shape}")
print(f"Numéricas ({len(numerical)}): {numerical}")
print(f"Categóricas ({len(categorical)}): {categorical}")


Dataset shape: (5000, 9)
Numéricas (4): ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']
Categóricas (4): ['lead_source', 'industry', 'employment_status', 'location']


## Q1. Moda de la columna 'industry'
¿Cuál es la observación más frecuente (moda) de `industry`?

In [2]:
# Pregunta 1
mode_industry = df["industry"].mode().iloc[0]
print(f"Moda de industry: {mode_industry}")


Moda de industry: technology


## Q2. Matriz de correlación para variables numéricas
¿Qué par de variables tiene la mayor correlación entre los candidatos?
- `interaction_count` y `lead_score`
- `number_of_courses_viewed` y `lead_score`
- `number_of_courses_viewed` y `interaction_count`
- `annual_income` y `interaction_count`

In [3]:
# Pregunta 2
corr_matrix = features[numerical].corr().abs()

candidate_pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count"),
]

for col1, col2 in candidate_pairs:
    print(f"{col1} y {col2}: {corr_matrix.loc[col1, col2]:.4f}")

best_pair = max(candidate_pairs, key=lambda pair: corr_matrix.loc[pair[0], pair[1]])
print(f"\nPar con mayor correlación: '{best_pair[0]}' y '{best_pair[1]}'")


interaction_count y lead_score: 0.9157
number_of_courses_viewed y lead_score: 0.7572
number_of_courses_viewed y interaction_count: 0.7216
annual_income y interaction_count: 0.1228

Par con mayor correlación: 'interaction_count' y 'lead_score'


## Partición de Datos (60% Train, 20% Val, 20% Test)

In [4]:
prepared_df = features.assign(**{TARGET: df[TARGET]})
df_full_train, df_test = train_test_split(prepared_df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

y_train = df_train.pop(TARGET).values
y_val = df_val.pop(TARGET).values
y_test = df_test.pop(TARGET).values

print(f"Train size: {len(df_train)}")
print(f"Val size:   {len(df_val)}")
print(f"Test size:  {len(df_test)}")


Train size: 3000
Val size:   1000
Test size:  1000


## Q3. Mutual Information con 'converted'
Calculado únicamente en el conjunto de entrenamiento.
¿Cuál variable categórica tiene el mayor puntaje de Mutual Information?

In [5]:
# Pregunta 3
mi_scores = {}
for col in categorical:
    mi_scores[col] = round(mutual_info_score(y_train, df_train[col]), 2)

for col, score in mi_scores.items():
    print(f"{col:<20}: {score:.2f}")

highest_mi = max(mi_scores, key=mi_scores.get)
print(f"\nVariable con mayor Mutual Information: '{highest_mi}'")


lead_source         : 0.03
industry            : 0.00
employment_status   : 0.02
location            : 0.00

Variable con mayor Mutual Information: 'lead_source'


## Q4. Regresión Logística (Modelo Completo)
Entrenar `LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)` con One-Hot Encoding (`DictVectorizer`).
Calcular el accuracy en validación redondeado a 2 decimales.

In [6]:
# Pregunta 4
def fit_logistic(train_feat, val_feat, c=1.0):
    dv = DictVectorizer(sparse=False)
    X_tr = dv.fit_transform(train_feat.to_dict(orient="records"))
    X_vl = dv.transform(val_feat.to_dict(orient="records"))

    model = LogisticRegression(solver="liblinear", C=c, max_iter=1000, random_state=42)
    model.fit(X_tr, y_train)

    preds = model.predict(X_vl)
    acc = accuracy_score(y_val, preds)
    return model, dv, acc

_, _, base_acc = fit_logistic(df_train, df_val, c=1.0)
print(f"Accuracy en validación: {base_acc:.2f} (exacto: {base_acc:.4f})")


Accuracy en validación: 0.65 (exacto: 0.6450)


## Q5. Eliminación de características (Feature Elimination)
Entrenar el modelo excluyendo sucesivamente cada variable candidata:
- `'lead_source'`
- `'number_of_courses_viewed'`
- `'interaction_count'`

¿Cuál característica genera la menor diferencia respecto al accuracy original?

In [7]:
# Pregunta 5
candidates_fe = ["lead_source", "number_of_courses_viewed", "interaction_count"]
diffs = {}

for col in candidates_fe:
    sub_train = df_train.drop(columns=[col])
    sub_val = df_val.drop(columns=[col])
    _, _, acc_sub = fit_logistic(sub_train, sub_val, c=1.0)
    diff = abs(base_acc - acc_sub)
    diffs[col] = diff
    print(f"Sin '{col}': Accuracy = {acc_sub:.4f} | Diferencia = {diff:.4f}")

least_impact = min(diffs, key=diffs.get)
print(f"\nCaracterística con menor diferencia: '{least_impact}'")


Sin 'lead_source': Accuracy = 0.6420 | Diferencia = 0.0030
Sin 'number_of_courses_viewed': Accuracy = 0.6430 | Diferencia = 0.0020


Sin 'interaction_count': Accuracy = 0.6010 | Diferencia = 0.0440

Característica con menor diferencia: 'number_of_courses_viewed'


## Q6. Ajuste de Regularización $C$
Evaluar $C \in [0.000001, 0.00001, 0.0001, 0.001]$. Redondear accuracy a 3 decimales.
Si hay empate, elegir el menor $C$.

In [8]:
# Pregunta 6
c_values = [0.000001, 0.00001, 0.0001, 0.001]
c_scores = {}

for c in c_values:
    _, _, acc = fit_logistic(df_train, df_val, c=c)
    c_scores[c] = round(acc, 3)
    print(f"C = {c:<8}: Accuracy = {acc:.4f} (redondeado: {round(acc, 3)})")

best_c = max(c_values, key=lambda c: (c_scores[c], -c))
print(f"\nMejor C: {best_c}")


C = 1e-06   : Accuracy = 0.5980 (redondeado: 0.598)


C = 1e-05   : Accuracy = 0.5980 (redondeado: 0.598)
C = 0.0001  : Accuracy = 0.6130 (redondeado: 0.613)


C = 0.001   : Accuracy = 0.6450 (redondeado: 0.645)

Mejor C: 0.001


### Resumen de Respuestas HW3:
- **Q1**: `technology`
- **Q2**: `interaction_count` and `lead_score`
- **Q3**: `lead_source`
- **Q4**: `0.65`
- **Q5**: `'number_of_courses_viewed'`
- **Q6**: `0.001`
